In [ ]:
ls

# HD Training

In [ ]:
#%tb

import importlib
import argparse
from omegaconf import OmegaConf
import os.path as osp
from datasets.inference_dataset import *
from datasets import *
import torch 

args = {'source': 'nuscenes', 'target': 'semantickitti', 'cluster_cfg': './cfg/clust_cfg/cluster_20.yaml', 
        'model_cfg': './cfg/model_cfg/kp_sk_infer.yaml', 'data_cfg_path': './cfg/data_cfg', 'subsample': 1, 
        'save_pred_path': '/root/main/3DLabelProp/results_3DLabelProp', 'train_hd': True, 'test_hd': True, 
        'hd_param': './cfg/hd_param.yaml'}

cfg = OmegaConf.create(args)
cluster_cfg = OmegaConf.load(cfg.cluster_cfg)
model_cfg = OmegaConf.load(cfg.model_cfg)
cfg = OmegaConf.merge(cfg,cluster_cfg,model_cfg)

if __name__ == "__main__":
    #Get info relative to the set
    if cfg.source == "semantickitti":
        source_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"semantic-kitti.yaml"))
        train_set = SemanticKITTI(source_data_cfg,'train')
    elif cfg.source == "nuscenes":
        source_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"nuscenes.yaml"))
        train_set = nuScenes(source_data_cfg,'train')
    else:
        raise  NameError('source dataset not supported')

    if cfg.target == "semantickitti":
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"semantic-kitti.yaml"))
        train_set_2 = SemanticKITTI(target_data_cfg,'train')
        target_set = SemanticKITTI(target_data_cfg,'valid')
    elif cfg.target == "nuscenes":
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"nuscenes.yaml"))
        train_set_2 = nuScenes(target_data_cfg,'train')
        target_set = nuScenes(target_data_cfg,'valid')
    elif cfg.target == "semanticposs":
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"semanticposs.yaml"))
        train_set_2 = SemanticPOSS(target_data_cfg,'train')
        target_set = SemanticPOSS(target_data_cfg,'valid')
    elif cfg.target == "semantickitti-nuscenes":
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"semantic-kitti-nuscenes.yaml"))
        train_set_2 = SemanticKITTI_Nuscenes(target_data_cfg,'train')
        target_set = SemanticKITTI_Nuscenes(target_data_cfg,'valid')
    elif "pandaset" in cfg.target:
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,cfg.target+".yaml"))
        train_set_2 = Pandaset(target_data_cfg,'train')
        target_set = Pandaset(target_data_cfg,'valid')
    
    else:
        raise  NameError('target dataset not supported')

    #Get info relative to the model
    if cfg.architecture.model == "KPCONV":
        module = importlib.import_module('models.kpconv.kpconv')
        model_information = getattr(module, cfg.architecture.type)()
        model_information.num_classes = train_set.get_n_label()
        model_information.ignore_label = -1
        model_information.in_features_dim = model_cfg.architecture.n_features
        model_information.train_hd = cfg.train_hd
        from models.kpconv_model import SemanticSegmentationModel
        module = importlib.import_module('models.kpconv.architecture')
        model_type = getattr(module, cfg.architecture.type)
        model = SemanticSegmentationModel(model_information,cfg,model_type)
    elif cfg.architecture.model == "SPVCNN":
        module = importlib.import_module('models.spvcnn.spvcnn')
        model_information = getattr(module, cfg.architecture.type)
        model_information.num_classes = train_set.get_n_label()
        model_information.ignore_label = -1
        model_information.in_features_dim = model_cfg.architecture.n_features
        from models.spvcnn_model import SemanticSegmentationSPVCNNModel
        model = SemanticSegmentationSPVCNNModel(model_information,cfg)
    else:
        raise  NameError('model not supported')
        
    # Get HD info
    if cfg.train_hd or cfg.test_hd:
        hd_cfg = OmegaConf.load(cfg.hd_param)
        cfg = OmegaConf.merge(cfg,hd_cfg) 
        from models.HD import OnlineHD
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        #device = torch.device("cpu")
        model_hd = OnlineHD(hd_cfg.n_features, hd_cfg.n_dimensions, hd_cfg.n_classes, epochs = hd_cfg.epochs, device=device)
        
    #print(cfg.hd_block_stop) #The parameters of hd are now part of cfg
    #try:
    #    ius, miu = valid_dataset.compute_results()
    #except:
    
    # Define the path for the "HD" folder
    hd_folder = os.path.join(cfg.save_pred_path, 'HD')
    
    # Define file names for saving the tensors
    weights_path = os.path.join(hd_folder, 'weights.pt')
    encoding_path = os.path.join(hd_folder, 'encoding.pt')

    # Check if the "HD" folder exists
    if not os.path.exists(hd_folder):
        os.makedirs(hd_folder)
        print(f"Folder 'HD' created at {hd_folder}")
    else:
        print(f"Folder 'HD' already exists at {hd_folder}")

    if cfg.train_hd:
        
        output_dataset = InferenceDataset(cfg,train_set,train_set_2, model, model_information, model_hd)
        
        output_dataset.compute_hd_dataset()

        # Save the tensors
        torch.save(model_hd.model.weight, weights_path)
        torch.save(model_hd.encoder.weight, encoding_path)

        print(f"Tensors saved in {hd_folder}")
    
    if cfg.test_hd:
        
        model_hd.model.weight = torch.load(weights_path)
        model_hd.encoder.weight = torch.load(encoding_path)
        
        output_dataset_2 = InferenceDataset(cfg, train_set, target_set, model, model_information, model_hd)
        
        output_dataset_2.compute_dataset()
        ius, miu = output_dataset_2.compute_results() # The results are already there?
        print(ius)
        print(miu)
    
    #ius, miu = output_dataset.compute_results() # The results are already there?
    #print(ius)
    #print(miu)

Model ready
Folder 'HD' already exists at /root/main/3DLabelProp/results_3DLabelProp/HD
Sequence:  ['00', '01', '02', '03', '04', '05', '06', '07', '09', '10']


Processing dataset semantickitti:   0%|                                                                                      | 0/10 [00:00<?, ?it/s]

Last:  004538.bin
Last:  004538



Processing dataset semantickitti:  10%|███████▊                                                                      | 1/10 [00:00<00:00,  9.52it/s]

Last:  001099.bin
Last:  001099


...

/root/anaconda3/envs/3DLabelProp/lib/python3.7/site-packages/torchhd/tensors/map.py:375: UserWarning: The norm of a vector is nearly zero, this could indicate a bug.
  "The norm of a vector is nearly zero, this could indicate a bug."


TrainHD?:  True
torch.Size([10790])
X_interme_enc:  torch.Size([10790, 64])
X_interme_enc:  torch.Size([10790, 128])
X_interme_enc:  torch.Size([9091, 128])
X_interme_enc:  torch.Size([9091, 256])
X_interme_enc:  torch.Size([9091, 256])
torch.Size([10790, 128])
torch.Size([128])
Encoded torch.Size([10790, 2000])
Weights torch.Size([19, 2000])
y torch.Size([10790, 19])
All equal? MAPTensor(True, device='cuda:0')
Infered: MAPTensor(119195, device='cuda:0')
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
[ 22209  22210  97884 ... 294766 296894 296895]
[[0.05718525 0.03485486 0.03485486 ... 0.06612621 0.05941304 0.05656418]
 [0.05259052 0.0328132  0.0328132  ... 0.06709727 0.06199865 0.05860984]
 [0.05329942 0.03269966 0.03269966 ... 0.06565131 0.06072585 0.05934041]
 ...
 [0.0589086  0.03819471 0.03819471 ... 0.06357871 0.06039531 0.05236018]
 [0.04901696 0.0303115  0.0303115  ... 0

/root/anaconda3/envs/3DLabelProp/lib/python3.7/site-packages/torchhd/tensors/map.py:375: UserWarning: The norm of a vector is nearly zero, this could indicate a bug.
  "The norm of a vector is nearly zero, this could indicate a bug."

Sequence: 08, subsample number 1/1:  70%|█████████████████████████████████████████████████████▏                      | 7/10 [01:14<00:33, 11.23s/it]

True
TrainHD?:  True
torch.Size([30274])
X_interme_enc:  torch.Size([30274, 64])
X_interme_enc:  torch.Size([30274, 128])
X_interme_enc:  torch.Size([11199, 128])
X_interme_enc:  torch.Size([11199, 256])
X_interme_enc:  torch.Size([11199, 256])
torch.Size([30274, 128])
torch.Size([128])
Encoded torch.Size([30274, 2000])
Weights torch.Size([19, 2000])
y torch.Size([30274, 19])
All equal? MAPTensor(True, device='cuda:0')
Infered: MAPTensor(442195, device='cuda:0')


/root/anaconda3/envs/3DLabelProp/lib/python3.7/site-packages/torchhd/tensors/map.py:375: UserWarning: The norm of a vector is nearly zero, this could indicate a bug.
  "The norm of a vector is nearly zero, this could indicate a bug."


# HD Forward

In [ ]:
#%tb

import importlib
import argparse
from omegaconf import OmegaConf
import os.path as osp
from datasets.inference_dataset import *
from datasets import *
import torch 

args = {'source': 'nuscenes', 'target': 'semantickitti', 'cluster_cfg': './cfg/clust_cfg/cluster_20.yaml', 
        'model_cfg': './cfg/model_cfg/kp_sk_infer.yaml', 'data_cfg_path': './cfg/data_cfg', 'subsample': 1, 
        'save_pred_path': '/root/main/3DLabelProp/results_3DLabelProp', 'train_hd': True, 'test_hd': False, 
        'hd_param': './cfg/hd_param.yaml'}

cfg = OmegaConf.create(args)
cluster_cfg = OmegaConf.load(cfg.cluster_cfg)
model_cfg = OmegaConf.load(cfg.model_cfg)
cfg = OmegaConf.merge(cfg,cluster_cfg,model_cfg)

if __name__ == "__main__":
    #Get info relative to the set
    if cfg.source == "semantickitti":
        source_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"semantic-kitti.yaml"))
        train_set = SemanticKITTI(source_data_cfg,'train')
    elif cfg.source == "nuscenes":
        source_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"nuscenes.yaml"))
        train_set = nuScenes(source_data_cfg,'train')
    else:
        raise  NameError('source dataset not supported')

    if cfg.target == "semantickitti":
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"semantic-kitti.yaml"))
        train_set_2 = SemanticKITTI(target_data_cfg,'train')
    elif cfg.target == "nuscenes":
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"nuscenes.yaml"))
        train_set_2 = nuScenes(target_data_cfg,'train')
    elif cfg.target == "semanticposs":
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"semanticposs.yaml"))
        train_set_2 = SemanticPOSS(target_data_cfg,'train')
    elif cfg.target == "semantickitti-nuscenes":
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"semantic-kitti-nuscenes.yaml"))
        train_set_2 = SemanticKITTI_Nuscenes(target_data_cfg,'train')
    elif "pandaset" in cfg.target:
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,cfg.target+".yaml"))
        train_set_2 = Pandaset(target_data_cfg,'train')
    
    else:
        raise  NameError('target dataset not supported')

    #Get info relative to the model
    if cfg.architecture.model == "KPCONV":
        module = importlib.import_module('models.kpconv.kpconv')
        model_information = getattr(module, cfg.architecture.type)()
        model_information.num_classes = train_set.get_n_label()
        model_information.ignore_label = -1
        model_information.in_features_dim = model_cfg.architecture.n_features
        model_information.train_hd = cfg.train_hd
        from models.kpconv_model import SemanticSegmentationModel
        module = importlib.import_module('models.kpconv.architecture')
        model_type = getattr(module, cfg.architecture.type)
        model = SemanticSegmentationModel(model_information,cfg,model_type)
    elif cfg.architecture.model == "SPVCNN":
        module = importlib.import_module('models.spvcnn.spvcnn')
        model_information = getattr(module, cfg.architecture.type)
        model_information.num_classes = train_set.get_n_label()
        model_information.ignore_label = -1
        model_information.in_features_dim = model_cfg.architecture.n_features
        from models.spvcnn_model import SemanticSegmentationSPVCNNModel
        model = SemanticSegmentationSPVCNNModel(model_information,cfg)
    else:
        raise  NameError('model not supported')
        
    # Get HD info
    if cfg.train_hd:
        hd_cfg = OmegaConf.load(cfg.hd_param)
        cfg = OmegaConf.merge(cfg,hd_cfg) 
        from models.HD import OnlineHD
        #device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        device = torch.device("cpu")
        model_hd = OnlineHD(hd_cfg.n_features, hd_cfg.n_dimensions, hd_cfg.n_classes, epochs = hd_cfg.epochs, device=device)
        
    #print(cfg.hd_block_stop) #The parameters of hd are now part of cfg

    output_dataset = InferenceDataset(cfg,train_set,train_set_2,model, model_information, model_hd)
    #try:
    #    ius, miu = valid_dataset.compute_results()
    #except:
    output_dataset.compute_dataset()
    ius, miu = output_dataset.compute_results() # The results are already there?
    print(ius)
    print(miu)